Import libraries and functions

In [97]:
# Import

# Import Python standard libraries
import json  # Import json module for parsing JSON data
import logging  # Import logging module for implementing event logging
from pathlib import Path  # From pathlib module import Path class for handling file paths
from typing import Union, Dict, Any, List, Optional  # From typing module import relevant classes for type hinting in function signatures
from urllib.parse import urlparse  # From urllib.parse module import urlparse function for identifying URLs

# Import external packages
import requests  # Import Requests library for downloading through URLs
import networkx as nx  # Import NetworkX package for network creation

Configure logging

In [98]:
# Configure logging

logging.basicConfig(  # Initiate configuration
    level=logging.DEBUG,  # Set minimum severity level
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'  # Set logging to display timestamp, logger name, severity level, and message
)

logger = logging.getLogger('affiliation_builder')  # Create package logger instance

Define build() function

In [99]:
# Define build() function

def build(  # Declare function
    # Define parameters    
    json_path: Union[str, Path],  # Take either string or Path object
    node_set_0_key: Optional[str],
    node_set_1_keys: Union[str, List[str]],  # Take either string or list of strings
    identifier_key: str,
    node_set_1_identifier_key: Optional[str] = None
) -> nx.Graph:  # Define return type
    """
    Build a bipartite affiliation network from JSON data.
    
    Creates a bipartite NetworkX graph where node set 0 (e.g., events) connects
    to node set 1 (e.g., human and organization participants in event) through
    affiliation relationships.
    All JSON fields are preserved as node attributes.
    
    Parameters
    ----------
    json_path : str or Path
        Path to JSON file or URL (http/https).
    node_set_0_key : str or None
        JSON key containing node-set-0 items (NetworkX bipartite=0). 
        Use None if JSON is a direct array without a wrapping key.
    node_set_1_keys : str or list of str
        JSON key(s) for affiliated entities (NetworkX bipartite=1) in each node-set-0
        item.
    identifier_key : str
        JSON key to use as unique identifier for node-set-0 item.
    node_set_1_identifier_key : str or None, default=None
        JSON key to use as identifier for node set 1 items when they are objects
        (rather than simple values).
        If None, assumes node set 1 items are simple values (strings/numbers).
        If provided, extracts this key from each entity object and preserves
        all other keys as node attributes.
                
    Returns
    -------
    networkx.Graph
        Bipartite graph with two node sets connected by affiliation edges.
        
    Examples
    --------
    >>> # Simple entities (strings)
    >>> G = build('events.json', 'events', 'participants', 'name')
    
    >>> # Complex entities (objects with metadata)
    >>> G = build(json_path='events.json',
    ...           node_set_0_key='events',
    ...           node_set_1_keys='participants',
    ...           identifier_key='name',
    ...           node_set_1_identifier_key='person_name')
    
    >>> # Direct array format
    >>> G = build('events.json', None, 'members', 'id')
    
    >>> # Multiple entity types with complex objects
    >>> G = build('events.json', 'events', ['persons', 'organizations'], 'name',
    ...           node_set_1_identifier_key='name')
    
    Notes
    -----
    Node set 0 has bipartite=0, node set 1 has bipartite=1. Edges only exist
    between nodes of different sets. Access node sets with:
    
        node_set_0 = {n for n, d in G.nodes(data=True) if d['bipartite'] == 0}
        node_set_1 = {n for n, d in G.nodes(data=True) if d['bipartite'] == 1}

    When node_set_1_identifier_key is specified, all other attributes from entity 
    objects are preserved as node attributes, enabling rich metadata analysis.

    """

    # Log function call and arguments
    logger.info(f"Affiliation network builder started")
    logger.info(f"JSON source: {json_path}")
    logger.info(f"Node-set-0 key: '{node_set_0_key}'")
    logger.info(f"Node-set-1 keys: {node_set_1_keys}")
    logger.info(f"Identifier key: '{identifier_key}'")
    logger.info(f"Node-set-1 identifier key: '{node_set_1_identifier_key}'")

    # =========================================================================
    # INPUT VALIDATION
    # =========================================================================
    # Validate input types
    
    # Validate node_set_1_keys type
    if not isinstance(node_set_1_keys, (str, list)):  # Check data type
        logger.error(
            f"Invalid type for node_set_1_keys detected: {type(node_set_1_keys)}"  # Log type error
        )
        raise TypeError(
            f"node_set_1_keys must be string or list, "
            f"but got {type(node_set_1_keys).__name__}"  # Raise type error
        )
    
    # Validate identifier_key type
    if not isinstance(identifier_key, str):
        logger.error(
            f"Invalid type for identifier_key detected: {type(identifier_key)}"
        )
        raise TypeError(
            f"identifier_key must be string, "
            f"but got {type(identifier_key).__name__}"
        )
    
    # Validate node_set_1_identifier_key type (if provided)
    if node_set_1_identifier_key is not None and not isinstance(node_set_1_identifier_key, str):
        logger.error(
            f"Invalid type for node_set_1_identifier_key detected: {type(node_set_1_identifier_key)}"
        )
        raise TypeError(
            f"node_set_1_identifier_key must be string or None, "
            f"but got {type(node_set_1_identifier_key).__name__}"
        )
    
    # Validate node_set_0_key type
    if node_set_0_key is not None and not isinstance(node_set_0_key, str):
        logger.error(
            f"Invalid type for node_set_0_key detected: {type(node_set_0_key)}"
        )
        raise TypeError(
            f"node_set_0_key must be string or None, "
            f"but got {type(node_set_0_key).__name__}"
        )
    
    logger.debug("Input validation passed")

    # =========================================================================
    # STEP 1: NORMALIZE node_set_1_keys TO A LIST
    # =========================================================================
    # Users can pass either a string or a list. Strings will be converted to
    # lists for later processing.
    # Example: 'members' becomes ['members']
    # Example: ['persons', 'organizations'] stays ['persons', 'organizations']
    
    if isinstance(node_set_1_keys, str):  # Detect string type
        node_set_1_keys = [node_set_1_keys]  # Convert to list
        logger.debug(f"Converting node_set_1_keys to list: {node_set_1_keys}")  # Log normalization

    # =========================================================================
    # STEP 2: LOAD JSON DATA (LOCAL FILE OR URL)
    # =========================================================================
    # Determine data source and load data.

    is_url = False  # Set local source as default

    # Check URL source
    if isinstance(json_path, str):  # Check string type
        parsed = urlparse(json_path)  # Parse URL
        is_url = parsed.scheme in ('http', 'https')  # Check URL scheme
    
    # Load from URL

    if is_url:
        logger.info(f"URL detected, downloading JSON from: {json_path}")  # Log URL detection

        # Load and parse JSON from URL

        try:
            response = requests.get(json_path, timeout=30)  # Send server request; set timeout to 30 seconds; return Response object
            response.raise_for_status()  # Raise exception for HTTP status code 4xx or 5xx
            data = response.json()  # Decode Response object as UTF-8 text; parse as JSON into Python list or dictionary
            logger.info(f"JSON successfully downloaded") # Log URL download
                        
        except requests.exceptions.Timeout:  # Catch timeout error
            logger.error(f"Request timed out")  # Log timeout error
            raise requests.exceptions.Timeout(
                f"Request timed out while accessing {json_path}"  # Re-raise timeout error with added context
            )
        
        except requests.exceptions.ConnectionError as e:
            logger.error(f"Connection error: {e}")  # Log ConnectionError specifics
            raise requests.exceptions.ConnectionError(
                f"Could not connect to {json_path}"
            )
        
        except requests.exceptions.HTTPError as e:  # Catch HTTP error raised by response.raise_for_status()
            logger.error(f"HTTP error: {e}")
            raise  # Re-raise without added context (URL already part of original error message)

        except json.JSONDecodeError as e:  # Catch JSON error
            logger.error(f"Downloaded content is not valid JSON")
            raise json.JSONDecodeError(
                f"Downloaded content is not valid JSON: {e.msg}",
                e.doc,
                e.pos
            )  # Re-raise with error message, document, and fail position information
        
        except requests.exceptions.RequestException as e:  # Catch any other error from Requests library
            logger.error(f"Unexpected error downloading from URL: {e}")
            raise

    # Load locally

    else:
        json_path = Path(json_path)  # Convert string to Path object (if necessary)
        logger.info(f"Local file path detected: {json_path}")  # Log file path detection

        # Check path exists
        if not json_path.exists():
            logger.error(f"File not found: {json_path}")
            raise FileNotFoundError(f"JSON file not found: {json_path}")
        
        # Check path points to file
        if not json_path.is_file():
            logger.error(f"Not a file: {json_path}")
            raise ValueError(
                f"Path must point to a file, not a directory: {json_path}"
            )
        
        # Load and parse JSON from file path

        try:
            with json_path.open('r', encoding='utf-8') as f:  # Open for reading as UTF-8; manage context
                data = json.load(f)  # Parse file object as JSON; return Python list or dictionary
            
            logger.info(f"Local JSON file successfully loaded")
            
        except json.JSONDecodeError as e:
            logger.error(f"Invalid JSON format in file")
            raise json.JSONDecodeError(
                f"File is not valid JSON: {e.msg}",
                e.doc,
                e.pos
            )
                
        except UnicodeDecodeError as e:  # Catch encoding error
            logger.error(f"File encoding error: {e}")
            raise ValueError(f"Unable to read file as UTF-8: {e}")
        
        except Exception as e:  # Catch any other error
            logger.error(f"Unexpected error loading JSON: {e}")
            raise

    # =========================================================================
    # STEP 3: VALIDATE JSON STRUCTURE AND EXTRACT ITEMS
    # =========================================================================
    # Handle both direct arrays and wrapped objects in JSON.
    
    if isinstance(data, list): # Check direct array format
        items = data  # Assign list without extraction
        logger.info(f"Direct array format with {len(items)} items detected")  # Log JSON structure
        
    elif isinstance(data, dict):  # Check wrapped object format
        if node_set_0_key is None:  # Check node-set-0 key is passed
            logger.error("Wrapped object format detected, but node_set_0_key is None")  # Log missing key error
            raise ValueError(
                f"JSON has wrapped object format. node_set_0_key is required to specify "
                f"key pointing to data. Available keys: {list(data.keys())}"
            )  # Raise missing key error with list of Python dictionary keys
        
        if node_set_0_key not in data: # Check node-set-0 key is valid
            logger.error(f"Key '{node_set_0_key}' not found")  # Log invalid key error
            raise ValueError(
                f"The key '{node_set_0_key}' was not found in the JSON data. "
                f"Available keys: {list(data.keys())}" 
            ) # Raise invalid key error

        if not isinstance(data[node_set_0_key], list):  # Check value of node-set-0 key is list
            logger.error(f"Value for '{node_set_0_key}' is not a list")  # Log invalid value error
            raise ValueError(
                f"The value for '{node_set_0_key}' must be a list, "
                f"but got {type(data[node_set_0_key]).__name__}"
            )  # Raise invalid value error
        
        items = data[node_set_0_key]  # Extract and assign list
        logger.info(f"Wrapped object format with {len(items)} items in '{node_set_0_key}' detected")  # Log successful extraction
                
    else:  # Check other JSON format
        logger.error(f"Unexpected data type: {type(data)}")  # Log invalid data type error
        raise ValueError(
            f"Expected Python list or dictionary from JSON, but got {type(data).__name__}"
        )  # Raise invalid data type error
        
    if len(items) == 0:  # Check for empty data
        logger.warning(f"No items found in the data")  # Log empty data warning (without raising error)
    
    
    # =========================================================================
    # STEP 4: CREATE BIPARTITE GRAPH
    # =========================================================================
    
    G = nx.Graph()  # Initialize undirected NetworkX graph
    logger.info("NetworkX graph initialized")  # Log graph initialization
    
    
    # =========================================================================
    # STEP 5: ITERATE THROUGH ITEMS AND BUILD NETWORK
    # =========================================================================

    for idx, item in enumerate(items):  # Return index-item tuples; unpack and loop through items

        # Check for JSON object structure
        if not isinstance(item, dict):
            logger.warning(
                f"Item at index {idx} not a dictionary, but {type(item).__name__}, skipping."
            )
            continue  # Continue loop with next item
                
        # Check for passed node-set-0 key in dictionary 
        if identifier_key not in item:
            logger.warning(
                f"Item at index {idx} missing '{identifier_key}' key, skipping. "
                f"Available keys: {list(item.keys())}"
            )
            continue
        
        node_id = item[identifier_key]  # Extract value of node-set-0 key as graph node ID
        
        # Validate graph node ID
        if node_id is None or node_id == "":
            logger.warning(
                f"Item at index {idx} has empty or None identifier, skipping"
            )
            continue

        # Check if graph node ID is hashable (for NetworkX node processing)
        try:
            hash(node_id)
        except TypeError:  # Raise TypeError if not hashable
            logger.warning(
                f"Item at index {idx} has unhashable identifier '{node_id}', skipping"
            )
            continue
        
        # Remove key of node-set-1 entities list to avoid redundancy with edges
        item_attrs = item.copy()
        for key in node_set_1_keys:
            item_attrs.pop(key, None)

        # Add node-set-0 nodes to graph from IDs
        G.add_node(node_id, bipartite=0, **item_attrs)  # Add node and add (only non-entity-related) attributes (also from unpacked item dictionary)
        logger.debug(f"Node-set-0 node '{node_id}' added")
        
        # Check for passed node-set-1 key(s) in dictionary
        for key in node_set_1_keys:  # Loop through list of keys
            if key not in item:
                logger.debug(f"Key '{key}' not found in item '{node_id}', skipping")
                continue  # Skip key that's missing in dictionary

            entities = item[key]  # Extract value of key - must be a list
            
            # Validate list data type of node-set-1 key value
            if not isinstance(entities, list):
                logger.warning(
                    f"Value for '{key}' in item '{node_id}' not a list, but {type(entities).__name__}, skipping"
                )
                continue

            for entity in entities:  # Loop through entities of list
                
                # Handle entities that are JSON objects

                if isinstance(entity, dict):  # Check for complex JSON structure
                    
                    # Check for user specification of identifier key
                    if node_set_1_identifier_key is None:
                        logger.warning(
                            f"Entity in '{key}' for item '{node_id}' is object, "
                            f"but node_set_1_identifier_key is None. Specify key "
                            f"to be used as identifier, skipping: {entity}"
                        )
                        continue

                    # Check for identifier key in entity
                    if node_set_1_identifier_key not in entity:
                        logger.warning(
                            f"Entity in '{key}' for item '{node_id}' missing "
                            f"'{node_set_1_identifier_key}' key, skipping: {entity}"
                        )
                        continue
                    
                    entity_id = entity[node_set_1_identifier_key]  # Extract value of node_set_1_identifier_key as graph node ID
                    
                    entity_attrs = entity.copy()  # Copy dictionary for graph node attributes

                else:
                    
                    # Handle entities that are simple JSON values
                    
                    entity_id = entity  # Get graph node ID
                    
                    entity_attrs = {}  # Create empty dictionary of graph node attributes

                # Validate graph node ID
                if entity_id is None or entity_id == "":
                    logger.warning(
                        f"Entity in '{key}' for item '{node_id}' has empty or None "
                        f"identifier, skipping: {entity}"
                    )
                    continue
                
                # Check if graph node ID is hashable
                try:
                    hash(entity_id)
                except TypeError:
                    logger.warning(
                        f"Entity ID '{entity_id}' in '{key}' for item '{node_id}' "
                        f"is not hashable, skipping"
                    )
                    continue

                # Add node-set-1 nodes to graph from IDs
                G.add_node(entity_id, bipartite=1, **entity_attrs)
                logger.debug(f"Node-set-1 node '{entity_id}' added")
                
                # Add edge (affiliation relation) between entity and item
                G.add_edge(node_id, entity_id)
                logger.debug(f"Edge between '{node_id}' and '{entity_id}' added")

    # =========================================================================
    # STEP 6: LOG SUMMARY AND RETURN
    # =========================================================================
                   

Test build

In [100]:
G = build(
    'https://codeberg.org/timofruehwirth/affiliation-builder/raw/branch/main/examples/example.json',
    'events',
    ['persons', 'organizations'],
    'name',
    node_set_1_identifier_key='name'
)

2025-11-13 16:26:35,201 - affiliation_builder - INFO - Affiliation network builder started
2025-11-13 16:26:35,202 - affiliation_builder - INFO - JSON source: https://codeberg.org/timofruehwirth/affiliation-builder/raw/branch/main/examples/example.json
2025-11-13 16:26:35,203 - affiliation_builder - INFO - Node-set-0 key: 'events'
2025-11-13 16:26:35,203 - affiliation_builder - INFO - Node-set-1 keys: ['persons', 'organizations']
2025-11-13 16:26:35,204 - affiliation_builder - INFO - Identifier key: 'name'
2025-11-13 16:26:35,204 - affiliation_builder - INFO - Node-set-1 identifier key: 'name'
2025-11-13 16:26:35,205 - affiliation_builder - DEBUG - Input validation passed
2025-11-13 16:26:35,205 - affiliation_builder - INFO - URL detected, downloading JSON from: https://codeberg.org/timofruehwirth/affiliation-builder/raw/branch/main/examples/example.json
2025-11-13 16:26:35,207 - urllib3.connectionpool - DEBUG - Starting new HTTPS connection (1): codeberg.org:443
2025-11-13 16:26:35,31